In [ ]:
from data_preparation import prepare_dataset, get_loss_weights
from graph_construction import build_graph
from train_and_evaluate import (
    train_and_evaluate,
    train_mlp
)
from models import (
    Transformer_Model,
    GCN_Model,
    GAT_Model,
    BaselineMLP,
    GoldenTransformer
)
from resources import (
    DATA_PATH,
    DEVICE,
    OUTPUT_TABLES_PATH,
    OUTPUT_FIGURES_PATH
)

EPOCHS = 500
PATIENCE = 30

In [2]:
import pandas as pd
import torch
import os
import numpy as np

In [3]:
x, y, pos_combined, pos_spatial, pos_temporal = prepare_dataset(DATA_PATH)
class_weights = get_loss_weights(y = y)

There are 10034 natural fires (56.44 %)
There are 7744 human made fires (43.56 %)


# STRUCTURAL ABLATION

In [ ]:
NUM_RUNS = 5

## Training with LogArea as only node feature

In [ ]:

graph_types = ['spatial', 'temporal', 'combined', 'multirelational']
k_values = [5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25]
architectures = ['GCN', 'GAT', 'TransformerConv']

results = []

print(f"\nStarting Comprehensive Ablation Study ({NUM_RUNS} runs per config)...")
for g_type in graph_types:
    for k in k_values:
        data = build_graph(g_type, k, pos_spatial, pos_temporal, pos_combined, x, y)
        in_dim = data.x.size(1) 
        edge_dim = data.edge_attr.size(1)

        for arch in architectures:
            run_f1s = []
            run_accs = []
            
            for run in range(NUM_RUNS):
                if arch == 'GCN':
                    model = GCN_Model(input_dim=in_dim, hidden_dim=16)
                elif arch == 'GAT':
                    model = GAT_Model(input_dim=in_dim, hidden_dim=16, edge_dim=edge_dim)
                elif arch == 'TransformerConv':
                    model = Transformer_Model(input_dim=in_dim, hidden_dim=16, edge_dim=edge_dim)

                f1, acc, _ = train_and_evaluate(model, data, DEVICE, class_weights, EPOCHS, PATIENCE)
                run_f1s.append(f1)
                run_accs.append(acc)

            avg_f1, std_f1 = np.mean(run_f1s), np.std(run_f1s)
            avg_acc, std_acc = np.mean(run_accs), np.std(run_accs)

            print(f"Graph: {g_type.upper():<10} | K: {k:<2} | Model: {arch:<15} | F1: {avg_f1:.4f} ± {std_f1:.4f}")
            
            results.append({
                'Graph': g_type,
                'K': k,
                'Model': arch,
                'F1': avg_f1,
                'F1_std': std_f1,
                'Accuracy': avg_acc,
                'Accuracy_std': std_acc
            })

results_topology_df = pd.DataFrame(results)
results_topology_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "results_topology_df.csv"))

## Training MLP baselines (spatial, temporal and combined)

In [ ]:
HIDDEN_UNITS = 16

t_mask = data.train_mask.to(DEVICE)
v_mask = data.test_mask.to(DEVICE)

x_mlp_spatial = torch.cat([x, pos_spatial], dim=1)           # [LogArea, Lat, Lon]
x_mlp_temporal = torch.cat([x, pos_temporal], dim=1)         # [LogArea, Days]
x_mlp_combined = torch.cat([x, pos_spatial, pos_temporal], dim=1) # [LogArea, Lat, Lon, Days]

mlp_configs = [
    ('MLP_Spatial', x_mlp_spatial, 3),
    ('MLP_Temporal', x_mlp_temporal, 2),
    ('MLP_Combined', x_mlp_combined, 4)
]

mlp_results = []
print(f"\n--- Running MLP Baselines ({NUM_RUNS} runs each) ---")

for name, input_tensor, in_features in mlp_configs:
    run_f1s = []
    run_accs = []
    
    for run in range(NUM_RUNS):
        model = BaselineMLP(in_features, HIDDEN_UNITS).to(DEVICE)
        f1, acc, _ = train_mlp(model, input_tensor, y, t_mask, v_mask, DEVICE, class_weights)
        run_f1s.append(f1)
        run_accs.append(acc)
    
    mlp_results.append({
        'Graph': name,
        'K': 0,
        'Model': 'MLP',
        'F1': np.mean(run_f1s),
        'F1_std': np.std(run_f1s),
        'Accuracy': np.mean(run_accs),
        'Accuracy_std': np.std(run_accs)
    })

results_mlp_df = pd.DataFrame(mlp_results)
mlp_topology_comparison_df = pd.concat([results_topology_df, results_mlp_df], ignore_index=True)

mlp_topology_comparison_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "mlp_topology_comparison.csv"))

## Getting result tables

In [4]:
mlp_topology_comparison_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "mlp_topology_comparison.csv"))
results_topology_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "results_topology_df.csv"))

In [5]:
def get_paper_ready_table(df, group_by_col):
    # 1. Find best based on Mean F1
    idx = df.groupby(group_by_col)['F1'].idxmax()
    table = df.loc[idx].copy()
    
    # 2. Create "Mean ± Std" strings
    table['F1 Score'] = table.apply(lambda r: f"{r['F1']:.4f} ± {r['F1_std']:.4f}", axis=1)
    table['Accuracy Score'] = table.apply(lambda r: f"{r['Accuracy']:.4f} ± {r['Accuracy_std']:.4f}", axis=1)
    
    return table

# --- ARCHITECTURAL COMPARISON ---
arch_table = get_paper_ready_table(mlp_topology_comparison_df, 'Model')
arch_table = arch_table[['Model', 'Graph', 'K', 'F1 Score', 'Accuracy Score']].sort_values(by='F1 Score', ascending=False)
print("\n--- BEST OF EACH ARCHITECTURE (5-RUN AVG) ---")
print(arch_table.to_string(index=False))

# --- TOPOLOGY COMPARISON ---
topo_table = get_paper_ready_table(mlp_topology_comparison_df, 'Graph')
topo_table = topo_table.rename(columns={'Graph': 'Topology Construction'})
topo_table = topo_table[['Topology Construction', 'Model', 'K', 'F1 Score', 'Accuracy Score']].sort_values(by='F1 Score', ascending=False)
print("\n--- BEST OF EACH TOPOLOGY (5-RUN AVG) ---")
print(topo_table.to_string(index=False))


--- BEST OF EACH ARCHITECTURE (5-RUN AVG) ---
          Model           Graph  K        F1 Score  Accuracy Score
TransformerConv multirelational 21 0.6708 ± 0.0053 0.6808 ± 0.0058
            GAT multirelational 17 0.6506 ± 0.0085 0.6552 ± 0.0100
            GCN multirelational 17 0.6119 ± 0.0344 0.6139 ± 0.0349
            MLP     MLP_Spatial  0 0.5949 ± 0.0176 0.6289 ± 0.0082

--- BEST OF EACH TOPOLOGY (5-RUN AVG) ---
Topology Construction           Model  K        F1 Score  Accuracy Score
      multirelational TransformerConv 21 0.6708 ± 0.0053 0.6808 ± 0.0058
             temporal TransformerConv 11 0.6553 ± 0.0037 0.6606 ± 0.0033
             combined             GAT 23 0.6085 ± 0.0062 0.6223 ± 0.0040
              spatial             GAT 15 0.6075 ± 0.0065 0.6142 ± 0.0094
          MLP_Spatial             MLP  0 0.5949 ± 0.0176 0.6289 ± 0.0082
         MLP_Combined             MLP  0 0.5352 ± 0.0051 0.5574 ± 0.0171
         MLP_Temporal             MLP  0 0.5280 ± 0.0334 0.5392 

## Plotting Topology vs BestMLP with all features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

def plot_wildfire_results(results_df, final_comparison_df, OUTPUT_FIGURES_PATH):
    if not os.path.exists(OUTPUT_FIGURES_PATH):
        os.makedirs(OUTPUT_FIGURES_PATH)

    # Set style
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.4)
    graph_order = ['spatial', 'temporal', 'combined', 'multirelational']
    architectures = ['GCN', 'GAT', 'TransformerConv']
    
    # ---------------------------------------------------------
    # 0. DATA PREPARATION
    # ---------------------------------------------------------
    mlp_subset = final_comparison_df[final_comparison_df['Model'] == 'MLP']
    if not mlp_subset.empty:
        best_mlp = mlp_subset.sort_values(by='F1', ascending=False).iloc[0]
        mlp_f1 = best_mlp['F1']
        mlp_std = best_mlp['F1_std']
        mlp_name = best_mlp['Graph'].replace('_', ' ')
    else:
        mlp_f1, mlp_std, mlp_name = 0.5, 0.0, "Baseline"

    # ---------------------------------------------------------
    # 1. PEAK F1 SCORE WITH ERROR BARS (FIXED MAPPING)
    # ---------------------------------------------------------
    plt.figure(figsize=(12, 6), dpi=300)
    
    # Extract peak for each (Graph, Model)
    idx = results_df.groupby(['Graph', 'Model'])['F1'].idxmax()
    peak_results = results_df.loc[idx].copy()
    
    # Ensure categories match the intended order
    peak_results['Graph'] = pd.Categorical(peak_results['Graph'], categories=graph_order, ordered=True)
    peak_results['Model'] = pd.Categorical(peak_results['Model'], categories=architectures, ordered=True)
    peak_results = peak_results.sort_values(['Graph', 'Model'])

    ax = sns.barplot(data=peak_results, x='Graph', y='F1', hue='Model', 
                     order=graph_order, palette='viridis', capsize=.05)
    
    # NEW ROBUST ERROR BAR MAPPING
    # We find the center of each bar by looking at the container groups
    for i, model_name in enumerate(architectures):
        # Get the heights and errors for this specific architecture across all topologies
        model_data = peak_results[peak_results['Model'] == model_name].set_index('Graph').reindex(graph_order)
        means = model_data['F1'].values
        errors = model_data['F1_std'].values
        
        # Get the x-positions for this specific hue group
        # Containers[i] corresponds to the i-th architecture in the 'hue'
        x_coords = [rect.get_x() + rect.get_width() / 2 for rect in ax.containers[i]]
        
        # Plot error bars for this specific architecture
        plt.errorbar(x=x_coords, y=means, yerr=errors, fmt='none', c='black', capsize=3, elinewidth=1)

    plt.axhline(mlp_f1, color='red', linestyle='--', linewidth=2, label=f'Best MLP ({mlp_name})')
    plt.axhspan(mlp_f1 - mlp_std, mlp_f1 + mlp_std, color='red', alpha=0.1)
    
    plt.ylim(min(results_df['F1'].min(), mlp_f1) - 0.08, max(results_df['F1'].max(), mlp_f1) + 0.12)
    plt.title("Peak Predictive Performance (Mean ± SD)", fontweight='bold')
    plt.legend(title="Architecture", loc='lower right', frameon=True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, '01_peak_f1_comparison.png'))
    plt.show()

    # ---------------------------------------------------------
    # 2. PERFORMANCE LIFT ANALYSIS
    # ---------------------------------------------------------
    plt.figure(figsize=(8, 7), dpi=300)
    best_gnn = results_df.sort_values(by='F1', ascending=False).iloc[0]
    #relative lift
    lift_val = ((best_gnn['F1'] - mlp_f1) / mlp_f1) * 100
    
    lift_df = pd.DataFrame({
        'Model': [f'Best MLP\n({mlp_name})', f"Proposed GNN\n({best_gnn['Model']})"],
        'F1 Score': [mlp_f1, best_gnn['F1']],
        'std': [mlp_std, best_gnn['F1_std']]
    })
    
    sns.barplot(data=lift_df, x='Model', y='F1 Score', palette=['#d9534f', '#5bc0de'])
    plt.errorbar(x=[0, 1], y=lift_df['F1 Score'], yerr=lift_df['std'], fmt='none', c='black', capsize=10, elinewidth=1.5)
    
    plt.text(0.5, (best_gnn['F1'] + mlp_f1)/2, f"Relative Lift:\n{lift_val:+.2f}%", 
             ha='center', va='center', fontsize=16, fontweight='bold', 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))

    plt.title("Forensic Attribution Performance Gain", fontweight='bold')
    plt.ylabel('Macro F1 Score')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, '02_topology_performance_lift.png'))
    plt.show()

    # ---------------------------------------------------------
    # 3. MULTIRELATIONAL SENSITIVITY HEATMAP
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 5), dpi=300)
    m_data = results_df[results_df['Graph'] == 'multirelational']
    if not m_data.empty:
        pivot_heat = m_data.pivot(index="Model", columns="K", values="F1")
        sns.heatmap(pivot_heat, annot=True, cmap="YlGnBu", fmt=".3f", cbar_kws={'label': 'Mean F1'})
        plt.title("Multirelational Sensitivity ($K$ Neighbors vs Architecture)", fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, '03_multirelational_heatmap.png'))
    plt.show()


# Run the plotting
plot_wildfire_results(results_topology_df, mlp_topology_comparison_df, OUTPUT_FIGURES_PATH)